# Predict Typologies for Real Buildings

This notebook uses the fine-tuned classifier to predict threshold typologies for real buildings (b1-b4).

**Pipeline:**
- Load fine-tuned classifier
- Load real building thresholds (b1-b4)
- Predict typology probabilities for each threshold
- Visualize and export results

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from config_classifier import get_classifier_config
from classifier_model import ThresholdClassifier
from dataset_classifier import create_building_dataloader

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

## Configuration

In [ ]:
# Paths
PANOS_DIR = r"C:\Users\shrua\OneDrive\Desktop\threshold project\threshold\data\panos_buildings"
CANDIDATES_DIR = r"C:\Users\shrua\OneDrive\Desktop\threshold project\threshold\data\candidates"
MODEL_PATH = '../output_classifier/checkpoints/classifier_best.pt'
OUTPUT_DIR = '../output_classifier/building_predictions'

# Buildings to predict
BUILDING_IDS = ['b1', 'b2', 'b3', 'b4']

# Typology names
TYPOLOGY_NAMES = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8']

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Panos: {PANOS_DIR}")
print(f"  Candidates: {CANDIDATES_DIR}")
print(f"  Model: {MODEL_PATH}")
print(f"  Buildings: {BUILDING_IDS}")
print(f"  Output: {OUTPUT_DIR}")

## Load Fine-tuned Classifier

In [ ]:
print("Loading fine-tuned classifier...")

# Load checkpoint
checkpoint = torch.load(MODEL_PATH, map_location=device)
config = checkpoint['config']

# Create model
model = ThresholdClassifier(config)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print(f"  Trained epoch: {checkpoint['epoch']+1}")
print(f"  Val accuracy: {checkpoint['val_acc']:.2f}%")
print()

## Load Building Data

In [ ]:
print("Loading building thresholds...")

building_loader, building_dataset = create_building_dataloader(
    panos_dir=PANOS_DIR,
    candidates_dir=CANDIDATES_DIR,
    building_ids=BUILDING_IDS,
    batch_size=16,
    num_workers=2
)

print(f"\n✓ Loaded {len(building_dataset)} threshold sequences from {len(BUILDING_IDS)} buildings")
print()

## Predict Typologies

In [ ]:
print("="*80)
print("PREDICTING TYPOLOGIES FOR REAL BUILDINGS")
print("="*80)
print()

# Collect predictions
predictions = []

with torch.no_grad():
    for batch in tqdm(building_loader, desc="Predicting"):
        imgs = batch['frames'].to(device)
        
        # Get predictions
        probs = model.predict_proba(imgs)  # [B, 8]
        preds = probs.argmax(dim=1)  # [B]
        
        # Store results
        for i in range(len(imgs)):
            predictions.append({
                'building_id': batch['building_id'][i],
                'threshold_id': batch['threshold_id'][i],
                'frame_indices': batch['frame_indices'][i].tolist(),
                'predicted_class': TYPOLOGY_NAMES[preds[i].item()],
                'confidence': probs[i, preds[i]].item(),
                **{f'{typ}_prob': probs[i, j].item() for j, typ in enumerate(TYPOLOGY_NAMES)}
            })

print(f"\n✓ Predictions complete! Total: {len(predictions)} thresholds")
print()

## Export Predictions

In [ ]:
# Convert to DataFrame
df_all = pd.DataFrame(predictions)

# Save all predictions
all_path = os.path.join(OUTPUT_DIR, 'all_predictions.csv')
df_all.to_csv(all_path, index=False)
print(f"✓ Saved all predictions to: {all_path}")
print()

# Save per-building
for building_id in BUILDING_IDS:
    df_building = df_all[df_all['building_id'] == building_id]
    if len(df_building) > 0:
        building_path = os.path.join(OUTPUT_DIR, f'{building_id}_predictions.csv')
        df_building.to_csv(building_path, index=False)
        print(f"✓ Saved {building_id}: {len(df_building)} thresholds → {building_path}")

print()
print("Prediction files created successfully!")

## Prediction Summary

In [ ]:
print("="*80)
print("PREDICTION SUMMARY")
print("="*80)
print()

for building_id in BUILDING_IDS:
    df_building = df_all[df_all['building_id'] == building_id]
    if len(df_building) == 0:
        continue
    
    print(f"{building_id.upper()} ({len(df_building)} thresholds):")
    
    # Count predictions per typology
    pred_counts = df_building['predicted_class'].value_counts()
    for typ in TYPOLOGY_NAMES:
        count = pred_counts.get(typ, 0)
        pct = count / len(df_building) * 100
        print(f"  {typ}: {count:2d} ({pct:5.1f}%)")
    
    # Average confidence
    avg_conf = df_building['confidence'].mean()
    print(f"  Avg confidence: {avg_conf:.3f}")
    print()

print("="*80)
print()

## Visualize Predictions by Building

In [ ]:
# Create bar chart for each building
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, building_id in enumerate(BUILDING_IDS):
    ax = axes[idx]
    df_building = df_all[df_all['building_id'] == building_id]
    
    if len(df_building) == 0:
        ax.text(0.5, 0.5, f'{building_id}\nNo data', 
                ha='center', va='center', fontsize=14)
        ax.set_xticks([])
        ax.set_yticks([])
        continue
    
    # Count predictions
    pred_counts = df_building['predicted_class'].value_counts().reindex(
        TYPOLOGY_NAMES, fill_value=0
    )
    
    # Plot
    bars = ax.bar(TYPOLOGY_NAMES, pred_counts.values, alpha=0.7, edgecolor='black')
    
    # Color bars by typology
    colors = plt.cm.tab10(np.linspace(0, 1, 10))[:8]
    for bar, color in zip(bars, colors):
        bar.set_facecolor(color)
    
    ax.set_xlabel('Typology', fontsize=11, fontweight='bold')
    ax.set_ylabel('Count', fontsize=11, fontweight='bold')
    ax.set_title(f'{building_id.upper()} - Predicted Typologies (n={len(df_building)})', 
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'building_predictions_summary.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print(f"Summary visualization saved to: {os.path.join(OUTPUT_DIR, 'building_predictions_summary.png')}")

## Visualize Sample Predictions

In [ ]:
# Show a few example predictions with probability distributions
print("="*80)
print("SAMPLE PREDICTIONS")
print("="*80)
print()

# Pick one example from each building
num_examples = min(4, len(BUILDING_IDS))
fig, axes = plt.subplots(num_examples, 2, figsize=(14, 4*num_examples))

if num_examples == 1:
    axes = axes.reshape(1, -1)

for idx, building_id in enumerate(BUILDING_IDS[:num_examples]):
    df_building = df_all[df_all['building_id'] == building_id]
    if len(df_building) == 0:
        continue
    
    # Pick first threshold
    sample = df_building.iloc[0]
    
    # Left: Probability bar chart
    ax = axes[idx, 0]
    probs = [sample[f'{typ}_prob'] for typ in TYPOLOGY_NAMES]
    colors = ['green' if typ == sample['predicted_class'] else 'gray' 
              for typ in TYPOLOGY_NAMES]
    
    bars = ax.barh(TYPOLOGY_NAMES, probs, color=colors, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Probability', fontsize=11, fontweight='bold')
    ax.set_ylabel('Typology', fontsize=11, fontweight='bold')
    ax.set_title(f"{sample['threshold_id']}\nPredicted: {sample['predicted_class']} (conf: {sample['confidence']:.3f})",
                fontsize=11, fontweight='bold')
    ax.set_xlim([0, 1])
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add probability labels
    for bar, prob in zip(bars, probs):
        if prob > 0.05:
            ax.text(prob, bar.get_y() + bar.get_height()/2,
                   f'{prob:.3f}',
                   ha='left', va='center', fontsize=9, 
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # Right: Frame indices info
    ax = axes[idx, 1]
    ax.axis('off')
    
    info_text = f"Building: {sample['building_id']}\n"
    info_text += f"Threshold ID: {sample['threshold_id']}\n"
    info_text += f"Frame indices: {sample['frame_indices']}\n\n"
    info_text += "Top 3 Predictions:\n"
    
    # Get top 3
    probs_array = np.array(probs)
    top3_indices = probs_array.argsort()[::-1][:3]
    for rank, idx_typ in enumerate(top3_indices, 1):
        typ = TYPOLOGY_NAMES[idx_typ]
        prob = probs_array[idx_typ]
        info_text += f"  {rank}. {typ}: {prob:.3f} ({prob*100:.1f}%)\n"
    
    ax.text(0.1, 0.5, info_text, fontsize=11, verticalalignment='center',
           family='monospace',
           bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_predictions.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSample predictions saved to: {os.path.join(OUTPUT_DIR, 'sample_predictions.png')}")

## Final Summary

In [ ]:
print("="*80)
print("FINAL SUMMARY")
print("="*80)
print()
print(f"Buildings analyzed: {len(BUILDING_IDS)}")
print(f"Total thresholds predicted: {len(predictions)}")
print()
print("Output files:")
print(f"  All predictions: {os.path.join(OUTPUT_DIR, 'all_predictions.csv')}")
for building_id in BUILDING_IDS:
    building_path = os.path.join(OUTPUT_DIR, f'{building_id}_predictions.csv')
    if os.path.exists(building_path):
        print(f"  {building_id}: {building_path}")
print(f"  Summary plot: {os.path.join(OUTPUT_DIR, 'building_predictions_summary.png')}")
print()

# Overall statistics
print("Overall prediction distribution:")
overall_counts = df_all['predicted_class'].value_counts()
for typ in TYPOLOGY_NAMES:
    count = overall_counts.get(typ, 0)
    pct = count / len(df_all) * 100
    print(f"  {typ}: {count:3d} ({pct:5.1f}%)")
print()

print(f"Average confidence: {df_all['confidence'].mean():.3f}")
print(f"Min confidence: {df_all['confidence'].min():.3f}")
print(f"Max confidence: {df_all['confidence'].max():.3f}")
print()
print("="*80)
print("\n✓ Prediction complete! Results saved to:", OUTPUT_DIR)